# HPO Organ Mapping — Synthetic Data Demonstration

> ⚠️ **Synthetic data demonstration only.**  
> All gene names, module assignments, and HPO term IDs in this notebook are entirely fabricated.  
> No real patient, clinical, or unpublished research data are present.

This notebook demonstrates the **phenotype-to-organ mapping** step of a gene module anatomogram pipeline.
It mirrors the analytical *structure* of the real research pipeline:
HPO term → organ key mapping → per-module organ percentage aggregation → gganatogram-ready CSV output.

**Simplification note:** The real pipeline maps HPO terms to organs via full ontology traversal
(BFS over the HPO DAG, using `hp.obo` parsed with `pronto`), finding the nearest ancestor HPO term
that belongs to a curated organ target set. That approach handles HPO term hierarchy correctly and
requires no internet connection but does require the ~10 MB `hp.obo` file and the `pronto` library.

This demo replaces ontology traversal with a **simplified keyword-based mapping** applied directly
to HPO term names. The keywords are derived from biological domain knowledge, not from any internal
pipeline configuration. The resulting organ assignments are illustrative — they capture the broad
organ-system signal present in the synthetic data but are not a clinically validated method.

**Dependencies:** `pandas` only. No internet access, no ontology file, no additional packages.

## § 0 — Imports

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
print("Imports OK.")

## § 1 — Load Input Data

Two files are required:

| File | Contents |
|------|----------|
| `gene_hpo_module_DEMO.csv` | Synthetic gene–HPO–module associations (columns: `gene_symbol`, `entrez_id`, `module_id`, `hpo_id`, `hpo_name`) |
| `HPO_Organ_Label_Mapping.csv` | Mapping from gganatogram organ keys to human-readable labels (columns: `organ_key`, `organ_label`) |

All gene names (`GENE_A1` … `GENE_C3`) and HPO term IDs are fabricated.
The HPO term *names* reflect real ontology vocabulary so that keyword-based mapping is meaningful,
but the assignments of those terms to these synthetic genes are not derived from any real dataset.

In [ ]:
df_gene_hpo = pd.read_csv(DATA_DIR / "gene_hpo_module_DEMO.csv", dtype=str)
df_organs   = pd.read_csv(DATA_DIR / "HPO_Organ_Label_Mapping.csv", dtype=str)

print(f"gene_hpo_module_DEMO: {df_gene_hpo.shape[0]} rows")
print(f"Columns: {list(df_gene_hpo.columns)}")
print()
print(f"HPO_Organ_Label_Mapping: {len(df_organs)} organ keys")
print(f"Columns: {list(df_organs.columns)}")
print()
print("Modules in demo data:", sorted(df_gene_hpo["module_id"].unique()))
print("Genes per module:")
print(df_gene_hpo.groupby("module_id")["gene_symbol"].nunique().to_string())

In [ ]:
# Build the set of valid organ keys from HPO_Organ_Label_Mapping.csv
# Only organ keys present in this file will appear in the output.
VALID_ORGAN_KEYS = set(df_organs["organ_key"].str.strip())
print(f"Valid organ keys ({len(VALID_ORGAN_KEYS)}): {sorted(VALID_ORGAN_KEYS)}")

## § 2 — Keyword-to-Organ Mapping

The real pipeline maps each HPO term to organ keys by traversing the HPO ontology DAG upward
from the term to the nearest ancestor that belongs to a curated organ target set.

This demo replaces that traversal with **keyword substring matching** on HPO term names.
Each entry in `KEYWORD_ORGAN_MAP` below is a `(keyword, [organ_keys])` pair:
- `keyword` is matched case-insensitively as a substring of the HPO term name.
- All matching keywords contribute their organ keys; a term may map to multiple organs.
- Only organ keys present in `HPO_Organ_Label_Mapping.csv` are retained.

Keywords are ordered from most specific to least specific within each organ group.
This ordering does not affect the result (all matches are unioned) but aids readability.

**Limitation:** Keyword matching cannot handle HPO terms whose names do not contain an obvious
anatomical keyword (e.g., `Abnormality of the spleen` would need `spleen` in the organ list).
Terms that match no keyword are logged as unmapped and excluded from the output — this is expected
behaviour and does not indicate a bug.

In [ ]:
# ---------------------------------------------------------------------------
# Keyword → organ key mapping
# Built from biological domain knowledge; not copied from any internal pipeline.
# ---------------------------------------------------------------------------
KEYWORD_ORGAN_MAP = [
    # ── Eye & Retina ────────────────────────────────────────────────────────
    ("rod-cone",         ["retina", "eye"]),
    ("retinal",          ["retina", "eye"]),
    ("retina",           ["retina", "eye"]),
    ("macular",          ["retina", "eye"]),
    ("foveal",           ["retina", "eye"]),
    ("optic",            ["eye", "retina"]),
    ("visual",           ["eye"]),
    ("ocular",           ["eye"]),
    ("choroid",          ["eye"]),
    ("iris",             ["eye"]),
    # ── Central & Peripheral Nervous System ─────────────────────────────────
    ("nervous system",   ["brain", "nerve", "spinal_cord", "cerebral_cortex"]),
    ("cerebral",         ["brain", "cerebral_cortex"]),
    ("brain",            ["brain", "cerebral_cortex"]),
    ("ataxia",           ["brain", "cerebral_cortex", "spinal_cord"]),
    ("seizure",          ["brain", "cerebral_cortex"]),
    ("epilep",           ["brain", "cerebral_cortex"]),
    ("cortical",         ["cerebral_cortex"]),
    ("spinal",           ["spinal_cord"]),
    ("neuropath",        ["nerve"]),
    # ── Muscle ──────────────────────────────────────────────────────────────
    ("musculoskeletal",  ["skeletal_muscle", "bone"]),
    ("hypotonia",        ["skeletal_muscle", "smooth_muscle"]),
    ("myopath",          ["skeletal_muscle"]),
    ("muscle",           ["skeletal_muscle", "smooth_muscle"]),
    # ── Skeletal ────────────────────────────────────────────────────────────
    ("skeletal",         ["bone", "skeletal_muscle"]),
    ("osteo",            ["bone"]),
    # ── Kidney ──────────────────────────────────────────────────────────────
    ("kidney",           ["kidney", "renal_cortex"]),
    ("renal",            ["kidney", "renal_cortex"]),
    ("nephro",           ["kidney", "renal_cortex"]),
    # ── Liver ───────────────────────────────────────────────────────────────
    ("liver",            ["liver"]),
    ("hepat",            ["liver"]),
    # ── Blood / Hematopoietic ────────────────────────────────────────────────
    ("blood",            ["bone_marrow"]),
    ("hematopoietic",    ["bone_marrow"]),
    ("anemia",           ["bone_marrow"]),
    # ── Heart & Vasculature ──────────────────────────────────────────────────
    ("cardiovascular",   ["heart", "left_ventricle"]),
    ("heart",            ["heart", "left_ventricle"]),
    ("cardiac",          ["heart", "left_ventricle"]),
    ("ventricle",        ["left_ventricle"]),
    ("cardiomyopath",    ["heart", "left_ventricle"]),
    # ── Skin ────────────────────────────────────────────────────────────────
    ("cutaneous",        ["skin"]),
    ("dermat",           ["skin"]),
    ("skin",             ["skin"]),
    # ── Pancreas ────────────────────────────────────────────────────────────
    ("pancrea",          ["pancreas"]),
    # ── Urinary ─────────────────────────────────────────────────────────────
    ("urinary",          ["urinary_bladder"]),
    ("bladder",          ["urinary_bladder"]),
    # ── Gastrointestinal ────────────────────────────────────────────────────
    ("intestin",         ["small_intestine"]),
    ("gastro",           ["small_intestine"]),
    ("coloni",           ["small_intestine"]),
    ("bowel",            ["small_intestine"]),
]

print(f"Keyword rules defined: {len(KEYWORD_ORGAN_MAP)}")

In [ ]:
def map_term_to_organs(term_name: str) -> set:
    """Return the set of valid organ keys matching a single HPO term name."""
    name_lower = term_name.lower()
    matched = set()
    for keyword, organ_keys in KEYWORD_ORGAN_MAP:
        if keyword in name_lower:
            for ok in organ_keys:
                if ok in VALID_ORGAN_KEYS:
                    matched.add(ok)
    return matched


# Verify each unique HPO term in the demo data and report coverage
unique_terms = df_gene_hpo[["hpo_id", "hpo_name"]].drop_duplicates()

term_mapping = {}
unmapped_terms = []
for _, row in unique_terms.iterrows():
    organs = map_term_to_organs(row["hpo_name"])
    term_mapping[row["hpo_id"]] = organs
    if not organs:
        unmapped_terms.append((row["hpo_id"], row["hpo_name"]))

print(f"Unique HPO terms in demo data: {len(unique_terms)}")
print(f"Terms mapped to ≥1 organ:      {len(unique_terms) - len(unmapped_terms)}")
print(f"Terms unmapped:                {len(unmapped_terms)}")

if unmapped_terms:
    print("\nUnmapped terms (excluded from output):")
    for hpo_id, hpo_name in unmapped_terms:
        print(f"  {hpo_id}: {hpo_name}")

print("\nFull term → organ mapping:")
for _, row in unique_terms.sort_values("hpo_id").iterrows():
    organs = term_mapping[row["hpo_id"]]
    tag = "✓" if organs else "✗ unmapped"
    print(f"  {tag}  {row['hpo_id']}  {row['hpo_name'][:45]:<45}  →  {sorted(organs)}")

## § 3 — Per-Module Organ Percentage Aggregation

For each module × organ pair we compute:

$$\text{value} = \frac{\text{number of genes in the module with} \geq 1 \text{ phenotype mapped to that organ}}{\text{total genes in the module}} \times 100$$

The numerator counts **distinct genes** (a gene contributes at most 1 to each organ, even if it has
multiple HPO terms all mapping to the same organ). This is the same aggregation logic used in the
real pipeline.

Organs with a value of 0 % (no genes in the module map to that organ) are omitted from the output —
the R visualization step fills all missing organs with 0 when building the full anatomy background.

In [ ]:
results = []

for module_id, mod_df in df_gene_hpo.groupby("module_id"):
    module_genes = mod_df["gene_symbol"].unique()
    module_size  = len(module_genes)

    # Build per-gene organ sets
    gene_organ_sets = {}
    for gene in module_genes:
        gene_terms = mod_df[mod_df["gene_symbol"] == gene]["hpo_name"].tolist()
        gene_organs = set()
        for term_name in gene_terms:
            gene_organs.update(map_term_to_organs(term_name))
        gene_organ_sets[gene] = gene_organs

    # Count genes per organ (each gene counted at most once per organ)
    organ_gene_counts = {}
    for gene_organs in gene_organ_sets.values():
        for organ in gene_organs:
            organ_gene_counts[organ] = organ_gene_counts.get(organ, 0) + 1

    for organ, count in sorted(organ_gene_counts.items()):
        results.append({
            "module_id": module_id,
            "organ":     organ,
            "value":     round(100.0 * count / module_size, 1),
        })

df_output = pd.DataFrame(results, columns=["module_id", "organ", "value"])

print(f"Output rows: {len(df_output)}")
print()
for mid, grp in df_output.groupby("module_id"):
    print(f"Module {mid} — {len(grp)} organs with value > 0:")
    print(grp.sort_values("value", ascending=False).to_string(index=False))
    print()

## § 4 — Write Output CSV

The output is written to `data/gganatogram_organs_input_DEMO.csv` with exactly the columns
`module_id, organ, value` required by `scripts/gganatomogram_Plot.Rmd`.

This file overwrites the previous hand-fabricated version so that the values are now derived
from the actual demo input data rather than being manually entered.

In [ ]:
output_path = DATA_DIR / "gganatogram_organs_input_DEMO.csv"
df_output.to_csv(output_path, index=False)
print(f"Written: {output_path.resolve()}")
print(f"Rows: {len(df_output)}, Columns: {list(df_output.columns)}")

## § 5 — Summary

The notebook has produced `gganatogram_organs_input_DEMO.csv` — the input file for the
`gganatomogram_Plot.Rmd` visualization step. Pass `MODULE_ID = 1` (or 2 or 3) when knitting
the Rmd to generate an anatomogram for that module.

---

**What this demo covers:**
- Loading a synthetic gene–HPO–module annotation table
- Keyword-based mapping of HPO term names to anatomical organ keys
- Per-module aggregation of the gene-level organ assignments into percentages

**What the real pipeline additionally does:**
- Parses the full HPO ontology (`hp.obo`, ~19,000 terms) using `pronto`
- Builds a term-ancestor index via BFS over the `is_a` DAG
- Maps each HPO term to organ keys by finding the shortest path to a curated organ target set —
  this handles term specificity correctly and covers the entire ontology, not just terms whose
  names happen to contain an obvious anatomical keyword
- Applies a minimum-percentage threshold before writing the output

---
*Part of the [Evolutionary Genomics & Multi-Omics Portfolio](https://github.com/Shalev-CompBio/Shalev-Evolutionary-Genomics-Portfolio)*